# In-Class Activity: AI Workflows and AI Agents
In this activity, we will use AI to take off quantities from foundation plans and answer questions about a construction project: first as a fixed workflow, then as an AI agent with tools and a skill.

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line. Paste your Gemini API key from MP5 into *api_key* (or leave it empty to use the Colab secret GEMINI_API_KEY).
#@markdown Without a key, every step runs with its default settings (stored outputs); editing a prompt, a tool description, or the skill needs the key.
api_key = "" #@param {type:"string"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp8_agents", "aec_agent"
FOLDERS = ["mp8_agents"]                      # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
try:
    import google.genai
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], capture_output=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_agent import lab
model = "gemini-3.5-flash-lite"
lab.setup(api_key=api_key, model=model)


## Part 1 · AI Workflows: Fixed Steps
A workflow splits a task into fixed steps, each done by an AI model or by code. We compare one prompt with a two-step workflow on a takeoff from five foundation plans.

![One prompt vs. a workflow](https://raw.githubusercontent.com/Haolan-Zhang/CEM4644/master/mp8_agents/figures/workflow.png)

In [ ]:
#@title ▶ Step 1a · Look at the Drawings { display-mode: "form" }
#@markdown Choose a drawing to see it with its three questions. Some plans leave one side undimensioned.
drawing = "Foundation plan 1" #@param ["Foundation plan 1", "Foundation plan 2", "Foundation plan 3", "Foundation plan 4", "Foundation plan 5"]
lab.show_drawing(drawing)


In [ ]:
#@title ▶ Step 1b · One Prompt: The AI Model Does Everything { display-mode: "form" }
#@markdown Run the cell: the AI model (Gemini) reads each drawing and computes every quantity in one reply.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """{problem}

End your reply with one final line per question, in this format:
{lines}"""
result_text = """**One prompt:** {right} of {n} quantities within tolerance of the ground truth (5 drawings)."""
table_text = """Ground truth vs. model output:"""
columns_text = """Drawing | Question | Quantity | Ground truth | Model output | Unit | Correct"""
lab.one_prompt(prompt_text=prompt_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 1c · Workflow: The AI Model Reads, Code Computes { display-mode: "form" }
#@markdown Run the cell: step 1, the AI model reads the length of every side; step 2, code adds them and applies the estimating conventions.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
read_prompt_text = """The attached foundation plan shows a CMU foundation wall; the dimensions are to the outside face of the wall. List the length of every side of the wall, going around the outside face. If a side is not dimensioned, work out its length from the other dimensions.
End your reply with one line, in this format:
SIDES: <length 1>, <length 2>, ... (in feet)"""
result_text = """**Workflow (AI model reads, code computes):** {right} of {n} quantities within tolerance of the ground truth (5 drawings)."""
steps_text = """Step 1 (AI model) and step 2 (code) for each drawing:"""
steps_columns_text = """Drawing | Sides read by the AI model (ft) | Total wall length (ft) | Ground truth (ft)"""
table_text = """Ground truth vs. workflow output:"""
columns_text = """Drawing | Question | Quantity | Ground truth | Workflow output | Unit | Correct"""
lab.workflow(read_prompt_text=read_prompt_text, result_text=result_text, steps_text=steps_text, steps_columns_text=steps_columns_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 1d · Compare One Prompt and the Workflow { display-mode: "form" }
#@markdown Run the cell after Steps 1b and 1c.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Quantities within tolerance of the ground truth (15 quantities, 5 drawings)"""
columns_text = """Method | Correct | Model calls"""
empty_text = """Run Steps 1b and 1c first."""
lab.compare_workflow(title_text=title_text, columns_text=columns_text, empty_text=empty_text)


### 📝 Report question 1
> From Step 1d: Report the number of quantities within tolerance for one prompt and for the workflow, and include the comparison chart.

> From Steps 1b and 1c: Which quantities does one prompt get wrong, and why? Did the AI model read every side correctly in Step 1 of the workflow, including the undimensioned ones?

> Which steps of a takeoff would you give to an AI model and which to code (or a spreadsheet) on a real project, and why?

## Part 2 · AI Agents: Tool Calling
An AI agent decides by itself which tool to call and when it has enough to answer. You choose its tools; it answers six questions about a practice project whose documents it can reach only through the tools.

![How an AI agent calls a tool](https://raw.githubusercontent.com/Haolan-Zhang/CEM4644/master/mp8_agents/figures/agent.png)

- **search_spec**: searches the specification.
- **calculator**: exact arithmetic.
- **lookup_log**: reads the submittal log or the schedule.

In [ ]:
#@title ▶ Step 2a · Look at the Project Documents { display-mode: "form" }
#@markdown Choose a document to see what the tools read.
document = "specification (Section 03 30 00)" #@param ["specification (Section 03 30 00)", "submittal log", "schedule"]
lab.show_document(document)


In [ ]:
#@title ▶ Step 2b · Run the Agent with the Tools You Choose { display-mode: "form" }
#@markdown Tick the tools and run the cell; each run is added to the table. Try no tools, each tool alone, and all three.
#@markdown UNKNOWN: the agent said it could not find the value. Wrong: it guessed.
search_spec = True #@param {type:"boolean"}
calculator = True #@param {type:"boolean"}
lookup_log = True #@param {type:"boolean"}
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
system_text = """You are an AI agent that answers questions about the CEM4644 practice project, a fictional building project. Project information (specification requirements, submittals, schedule) is available only through your tools. Do not guess a project value: if you cannot find it, answer UNKNOWN.
End your reply with the final answer on its own line, in this format: FINAL: <answer>
If the question asks for several quantities, give one FINAL line per quantity, for example: FINAL: Q1 = 832 CMU"""
search_spec_text = """Searches the project specification (Section 03 30 00, cast-in-place concrete) and returns the lines that best match the search words, for example the requirements for one element."""
calculator_text = """Evaluates an arithmetic expression exactly and returns the result. Supports + - * / ** ( ), ceil, floor, round, sqrt, min and max, for example: ceil(212 / 48)."""
lookup_log_text = """Returns the rows of a project log that contain the search text: log = submittals (the submittal log: status and the mix of each submittal) or schedule (the schedule: quantity and daily output of each activity)."""
result_text = """**Run {run} · tools: {tools}**
**Correct:** {right} of {n}   **UNKNOWN:** {unknown}   **Wrong:** {wrong}"""
none_text = """no tools"""
table_text = """Ground truth vs. agent output (this run):"""
columns_text = """Question | Ground truth | Agent output | Result | Tools called"""
runs_text = """All runs so far:"""
runs_columns_text = """Run | Tools | Correct | UNKNOWN | Wrong"""
lab.agent_tools(search_spec, calculator, lookup_log, system_text=system_text, search_spec_text=search_spec_text, calculator_text=calculator_text, lookup_log_text=lookup_log_text, result_text=result_text, none_text=none_text, table_text=table_text, columns_text=columns_text, runs_text=runs_text, runs_columns_text=runs_columns_text)


In [ ]:
#@title ▶ Step 2c · Inspect the Tool Calls { display-mode: "form" }
#@markdown Choose a question to see, for each run of Step 2b, the tools the agent called, with their inputs and outputs.
question = "P5. Does the mix in submittal 03 30 00-02 meet this project's maximum w/cm for its element? Answer YES or NO." #@param ["P1. What is the maximum water-cementitious ratio (w/cm) for the foundation walls on this project?", "P2. A continuous footing is 187 ft long, 2 ft 7 in wide, and 1 ft 2 in deep. How many cubic yards of concrete are needed, including 7 % waste?", "P3. What is the review status of submittal 03 30 00-03?", "P4. How many working days are scheduled to place the slab-on-ground: the activity's quantity divided by its daily output, rounded up to a whole day?", "P5. Does the mix in submittal 03 30 00-02 meet this project's maximum w/cm for its element? Answer YES or NO.", "P6. Does the target slump of the mix in submittal 03 30 00-05 meet this project's maximum slump for its element? Answer YES or NO."]
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
run_text = """**Run {run}**
tools: {tools}"""
question_label_text = """question"""
call_label_text = """tool call"""
output_label_text = """tool output"""
answer_label_text = """answer"""
empty_text = """No runs yet: run Step 2b first."""
lab.show_trace(question, run_text=run_text, question_label_text=question_label_text, call_label_text=call_label_text, output_label_text=output_label_text, answer_label_text=answer_label_text, empty_text=empty_text)


### 📝 Report question 2
> From Step 2b: Include the table of runs. For each tool, report which questions it was needed for.

> From Step 2c: For question P5, describe the agent's steps with all three tools. Without the tools it needed, did the agent answer UNKNOWN or guess?

> Which is more concerning on a real project, an agent that answers UNKNOWN or one that guesses, and how would you check an agent's answer?

## Part 3 · Code Execution: The Agent Builds Its Own Tool
In Part 2 the agent chose from tools written for it. Here HokieAI writes and runs its own code, and the result is something you can see and use: a chart or a calculator. Each result is checked against the ground truth, because code that runs can still give wrong numbers.

![Tool calling vs. code execution](https://raw.githubusercontent.com/Haolan-Zhang/CEM4644/master/mp8_agents/figures/codeexec.png)

In [ ]:
#@title ▶ Step 3a · Look-Ahead Schedule: A Gantt Chart { display-mode: "form" }
#@markdown Run the cell and follow the steps: HokieAI computes each activity's duration and start day, draws a Gantt chart, and marks in red the concrete placements whose mix design is not approved.
#@markdown Look-ahead schedule: the near-term part of the project schedule that the field team plans from.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Use your Python tool to make a look-ahead schedule from the two attached files, schedule.csv and submittal_log.csv.
1. For each activity, the duration in working days = quantity / daily output, rounded up to a whole day.
2. {sequence}
3. Draw a Gantt chart: one horizontal bar per activity, from its start day to its finish day, with working days on the x-axis. Color a bar red if the activity places concrete (its description starts with "Place") and the mix-design submittal for its element in submittal_log.csv is not Approved; color the other bars gray.
4. After the chart, write the schedule as plain text in your reply (not inside the chart image), one line per activity, in this format (the last column: YES if the bar is red, otherwise NO):
A-110 | <start day> | <finish day> | <YES or NO>"""
steps_text = """1. Download {files} with the buttons below and copy the prompt.
2. Open {url} (sign in with your VT account), start a new chat, attach both files, and paste the prompt. Look at the chart and open the code HokieAI wrote.
3. Copy HokieAI's output with the **copy button** below its reply, and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**Look-ahead schedule:** {right} of {n} activities with the correct start day, finish day, and flag."""
chart_title_text = """Ground truth: the look-ahead schedule (red = the mix design for the element is not approved)"""
axis_text = """working day"""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Activity | Ground truth (start – finish, red) | HokieAI's output | Correct"""
lab.lookahead(prompt_text=prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, chart_title_text=chart_title_text, axis_text=axis_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 3b · Build a Takeoff Calculator { display-mode: "form" }
#@markdown Run the cell and follow the steps: HokieAI builds a calculator web page (an artifact); you test it with the three cases and enter its outputs.
version = "without the skill"
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Build an interactive calculator as an artifact (a web page I can use): a takeoff calculator for a CMU foundation wall.
Inputs: the lengths of all sides of the wall, measured on the outside face (ft, separated by commas); the wall height (ft); CMU waste (%); grout per fully grouted CMU (ft³); grout waste (%); the spacing of the grouted cells (in on center).
Outputs: the total wall length (ft), the number of 8 in x 16 in CMUs to purchase, grout for all cells (CY), and grout for only the cells at the given spacing (CY).
Show the formula used for each output."""
steps_text = """1. Copy the prompt below, open {url} (sign in with your VT account), start a new chat, and paste the prompt.
2. Open the calculator HokieAI builds (the artifact) and type in each test case below.
3. Enter your calculator's outputs in the box below, one line per case, and click *Score my calculator*."""
cases_text = """Test cases (type each one into your calculator):"""
cases_columns_text = """Case | Sides (ft) | Height (ft) | CMU waste (%) | Grout per CMU (ft³) | Grout waste (%) | Grouted cells (in O.C.)"""
button_text = """Score my calculator"""
reply_text = """C1 | <CMUs> | <grout, all cells (CY)> | <grout at the spacing (CY)>
C2 | ...
C3 | ..."""
prompt_label_text = """prompt"""
reply_label_text = """results"""
result_text = """**Your calculator ({version}):** {right} of {n} outputs within tolerance of the ground truth."""
table_text = """Ground truth vs. your calculator:"""
columns_text = """Case | Quantity | Ground truth | Your calculator | Unit | Correct"""
runs_text = """Your calculators so far:"""
runs_columns_text = """Calculator | Outputs correct"""
lab.calculator(version, prompt_text=prompt_text, steps_text=steps_text, cases_text=cases_text, cases_columns_text=cases_columns_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text, runs_text=runs_text, runs_columns_text=runs_columns_text)


### 📝 Report question 3
> From Step 3a: Include HokieAI's Gantt chart and the score. Which activities did it mark red, and did any start or finish day differ from the ground truth? Open the code it wrote: which step went wrong, if any?

> From Step 3b: Include a screenshot of your calculator and the score on the three test cases. Which output was wrong, and which formula did the calculator use for it?

> When would you let an agent write and run code for a project task, and what would you check before using its output?

## Part 4 · Skills
A skill is a saved set of instructions with a name and a description. The agent sees only the descriptions and loads a skill's instructions when a task matches; a prompt is read every time.

![A prompt vs. a skill](https://raw.githubusercontent.com/Haolan-Zhang/CEM4644/master/mp8_agents/figures/skills.png)

| | Prompt | Skill |
|---|---|---|
| Written | in each message | once, saved with a name and a description |
| Read by the model | every time | only when the task matches the description |
| Example | the conventions pasted above every takeoff question | *cmu-foundation-takeoff*, loaded for CMU questions only |

In [ ]:
#@title ▶ Step 4a · Run the Agent with and without the Skill { display-mode: "form" }
#@markdown Run the cell with *use_skill* ticked and unticked: the agent answers two takeoff questions (T1, T2) and one project question (P3).
#@markdown Then make the skill's description vague (for example, *Useful notes.*) and run again: is the skill still loaded?
use_skill = True #@param {type:"boolean"}
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
system_text = """You are an AI agent that answers questions about the CEM4644 practice project, a fictional building project. Project information (specification requirements, submittals, schedule) is available only through your tools. Do not guess a project value: if you cannot find it, answer UNKNOWN.
End your reply with the final answer on its own line, in this format: FINAL: <answer>
If the question asks for several quantities, give one FINAL line per quantity, for example: FINAL: Q1 = 832 CMU"""
calculator_text = """Evaluates an arithmetic expression exactly and returns the result. Supports + - * / ** ( ), ceil, floor, round, sqrt, min and max, for example: ceil(212 / 48)."""
lookup_log_text = """Returns the rows of a project log that contain the search text: log = submittals (the submittal log: status and the mix of each submittal) or schedule (the schedule: quantity and daily output of each activity)."""
load_skill_text = """Loads the full instructions of one of the available skills by its name. Load a skill when its description matches the task."""
skills_list_text = """Available skills (load one with load_skill when its description matches the task):
- {name}: {description}"""
card_text = """**The skill** · **name:** {name}
**description:** {description}
**body:**"""
skill_name_text = """cmu-foundation-takeoff"""
skill_description_text = """Estimating conventions for CMU foundation walls: total wall length, number of CMUs, and grout volume. Use it for any CMU or grout quantity question."""
skill_body_text = """# CMU foundation takeoff: estimating conventions
- Total wall length = the sum of the outside dimensions of all sides (no deduction at the corners).
- Wall area = total wall length x wall height.
- One 8 in x 16 in (nominal) CMU covers 0.889 ft² of wall.
- CMUs to purchase = wall area / 0.889 ft², plus waste, rounded up to a whole CMU.
- Grout, all cells = CMUs before waste x grout per CMU, converted to cubic yards (27 ft³ = 1 CY), plus grout waste.
- Cells are 8 in apart, so grouting the cells at S in on center fills 8/S of the cells: grout = grout for all cells x 8 / S."""
result_text = """**Run {run} · skill: {skill}**
**Quantities correct:** {right} of {n}   **Skill loaded for:** {loaded}
**Tool calls:** {calls}   **Model calls:** {model_calls}"""
no_skill_text = """not available"""
none_loaded_text = """no question"""
table_text = """Ground truth vs. agent output (this run):"""
columns_text = """Question | Quantity | Ground truth | Agent output | Unit | Correct | Skill loaded"""
runs_text = """All runs so far:"""
runs_columns_text = """Run | Skill | Quantities correct | Skill loaded for | Tool calls | Model calls"""
lab.agent_skill(use_skill, system_text=system_text, calculator_text=calculator_text, lookup_log_text=lookup_log_text, load_skill_text=load_skill_text, skills_list_text=skills_list_text, card_text=card_text, skill_name_text=skill_name_text, skill_description_text=skill_description_text, skill_body_text=skill_body_text, result_text=result_text, no_skill_text=no_skill_text, none_loaded_text=none_loaded_text, table_text=table_text, columns_text=columns_text, runs_text=runs_text, runs_columns_text=runs_columns_text)


In [ ]:
#@title ▶ Step 4b · Create the Skill in HokieAI { display-mode: "form" }
#@markdown Create the same skill in HokieAI, then send the takeoff questions without and with it.
skill = "with the skill" #@param ["without the skill", "with the skill"]
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """{problems}

End your reply with one final line per question, in this format:
{lines}"""
steps_text = """1. In HokieAI ({url}), open **Account → My Skills → Create Skill**. Copy the skill's name, description, and body from Step 4a into the form, switch *Always On* off, and click *Create Skill*.
2. Choose *with the skill* or *without the skill* above and run this cell. Start a new chat; for *with the skill*, turn the skill on from the chat menu.
3. Copy the prompt below, paste it into the chat, and send.
4. Copy HokieAI's output with the **copy button** below its reply, paste it into the box below, and click *Score HokieAI output*."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**HokieAI, {skill}:** {right} of {n} quantities within tolerance of the ground truth."""
table_text = """Ground truth vs. HokieAI's output:"""
columns_text = """Question | Quantity | Ground truth | HokieAI's output | Unit | Correct"""
lab.hokieai_skill(skill, prompt_text=prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 4c · Rebuild the Calculator with the Skill { display-mode: "form" }
#@markdown Run the cell and follow the steps: the same calculator prompt as in Step 3b, this time with the skill turned on. The table compares both calculators.
version = "with the skill"
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Build an interactive calculator as an artifact (a web page I can use): a takeoff calculator for a CMU foundation wall.
Inputs: the lengths of all sides of the wall, measured on the outside face (ft, separated by commas); the wall height (ft); CMU waste (%); grout per fully grouted CMU (ft³); grout waste (%); the spacing of the grouted cells (in on center).
Outputs: the total wall length (ft), the number of 8 in x 16 in CMUs to purchase, grout for all cells (CY), and grout for only the cells at the given spacing (CY).
Show the formula used for each output."""
steps_text = """1. In HokieAI ({url}), start a new chat and turn on the skill of Step 4b (*cmu-foundation-takeoff*) from the chat menu.
2. Copy the prompt below (the same as in Step 3b), paste it, and open the new calculator.
3. Type in the same test cases, enter the outputs in the box below, and click *Score my calculator*."""
cases_text = """Test cases (type each one into your calculator):"""
cases_columns_text = """Case | Sides (ft) | Height (ft) | CMU waste (%) | Grout per CMU (ft³) | Grout waste (%) | Grouted cells (in O.C.)"""
button_text = """Score my calculator"""
reply_text = """C1 | <CMUs> | <grout, all cells (CY)> | <grout at the spacing (CY)>
C2 | ...
C3 | ..."""
prompt_label_text = """prompt"""
reply_label_text = """results"""
result_text = """**Your calculator ({version}):** {right} of {n} outputs within tolerance of the ground truth."""
table_text = """Ground truth vs. your calculator:"""
columns_text = """Case | Quantity | Ground truth | Your calculator | Unit | Correct"""
runs_text = """Your calculators so far:"""
runs_columns_text = """Calculator | Outputs correct"""
lab.calculator(version, prompt_text=prompt_text, steps_text=steps_text, cases_text=cases_text, cases_columns_text=cases_columns_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text, runs_text=runs_text, runs_columns_text=runs_columns_text)


### 📝 Report question 4
> From Step 4a: Report the quantities correct with and without the skill, and the questions the skill was loaded for. Did the agent load the skill for P3? What happened with a vague description?

> From Steps 4b and 4c: Report HokieAI's quantities correct without and with the skill, and your two calculators' scores. Which outputs did the skill fix?

> What estimating conventions or company rules from your own experience would you write as a skill, and what would its description say?

## Part 5 · Your Own HokieAI Agent
A HokieAI agent is instructions plus knowledge sources (files it can read). Build one with the project files and ask it the six questions of Part 2.

In [ ]:
#@title ▶ Step 5a · Create Your HokieAI Agent { display-mode: "form" }
#@markdown Run the cell and follow the steps; edit the instructions in the box before pasting them.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
instructions_text = """You answer questions about the CEM4644 practice project, a fictional building project, using only the attached files: the specification excerpt (spec_03_30_00.txt), the submittal log (submittal_log.csv), and the schedule (schedule.csv). Show each value you look up and each calculation. Do not guess a project value: if the files do not contain it, answer UNKNOWN."""
create_steps_text = """1. Download the three project files with the buttons below.
2. In HokieAI ({url}), open **Explore Agents** and click *Create* next to *My Agents*; enter a name, for example *CEM4644 project agent*.
3. Under **Agent Instructions → Specific**, paste the instructions below.
4. Under **Knowledge Sources → Add → Upload Files**, upload the three files.
5. Click *Create*."""
lab.create_agent(instructions_text=instructions_text, create_steps_text=create_steps_text)


In [ ]:
#@title ▶ Step 5b · Ask Your HokieAI Agent { display-mode: "form" }
#@markdown Run the cell after Step 5a and follow the steps.
# The wording the notebook shows (prompts, tool descriptions, skills, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
prompt_text = """Answer the questions below. Reply with one line per question and nothing else, in this format:
P1 | <answer>

Questions:
{questions}"""
steps_text = """1. Open your agent in HokieAI ({url}) and start a new chat with it.
2. Copy the prompt below, paste it into the chat, and send.
3. Copy the agent's output with the **copy button** below its reply, and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth and with the Gemini agent's runs in Step 2b."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**Your HokieAI agent:** {right} of {n} correct, {unknown} UNKNOWN, {wrong} wrong."""
table_text = """Ground truth vs. your HokieAI agent's output:"""
columns_text = """Question | Ground truth | HokieAI agent output | Result"""
compare_text = """Your HokieAI agent vs. the Gemini agent's runs in Step 2b:"""
compare_columns_text = """Agent | Correct | UNKNOWN | Wrong"""
lab.hokieai_agent(prompt_text=prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text, compare_text=compare_text, compare_columns_text=compare_columns_text)


### 📝 Report question 5
> From Step 5b: Report your agent's correct, UNKNOWN, and wrong answers, next to the Gemini agent's run with all three tools, and include the table.

> Which instructions did you change, and what changed in the output? For one wrong answer, explain from the output which value or calculation went wrong.

> What would you need to see before you let an agent answer project questions for your team?

### Data and model sources
- Practice project (specification excerpt, submittal log, schedule) and foundation plans: made for CEM4644 (fictional, not for construction); the specification excerpt follows the format of UFGS 03 30 00, U.S. Department of Defense, public domain, https://www.wbdg.org/dod/ufgs.
- AI model: Gemini (gemini-3.5-flash-lite), Google, via the Gemini API with your own key.
- Generative AI: GPT models via HokieAI (Virginia Tech).